In [ ]:
import sys
if 'google.colab' in sys.modules:
    import subprocess
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'tesseract-ocr-por'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'opencv-python-headless', 'pytesseract', 'pandas'], check=True)
    print('Dependências instaladas no Colab.')
else:
    print('Ambiente local: utilize requirements.txt e instale o Tesseract com idioma por.')


In [ ]:
from pathlib import Path
import os
import re
import time
import unicodedata

import cv2
import pandas as pd
import pytesseract
from pytesseract import Output

SALVAR_TEXTO_COMPLETO = False

NOMES_PASTAS = {
    'rg_antigo': 'rg_antigos',
    'rg_novo': 'rg_novos',
    'invalido': 'invalidos',
}
EXTENSOES_IMAGENS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

CANDIDATOS_PASTA_BASE = [
    Path(os.environ['PASTA_IMAGENS']) if os.environ.get('PASTA_IMAGENS') else None,
    Path('arquivos'),
    Path('dataset_rgs/dataset_rgs'),
    Path('Projeto_Suzana/dataset_rgs/dataset_rgs'),
    Path('/content/arquivos'),
    Path('/content/dataset_rgs/dataset_rgs'),
    Path('/content/Projeto_Suzana/dataset_rgs/dataset_rgs'),
    Path('../dataset_rgs/dataset_rgs'),
]

def localizar_pasta_imagens():
    for base in CANDIDATOS_PASTA_BASE:
        if base is not None and all((base / nome).is_dir() for nome in NOMES_PASTAS.values()):
            return base
    raise FileNotFoundError(
        'Pastas de imagens não encontradas. Extraia Projeto_Suzana.zip no Colab '
        'ou configure PASTA_IMAGENS com o diretório que contém '
        'rg_antigos, rg_novos e invalidos.'
    )

PALAVRAS_CHAVE = (
    'BRASIL', 'REPUBLICA', 'IDENTIFICACAO', 'CPF',
    'NASCIMENTO', 'FILIACAO', 'REGISTRO GERAL',
    'CARTEIRA DE IDENTIDADE',
)

def normalizar_texto(texto):
    texto = unicodedata.normalize('NFD', texto.upper())
    return ''.join(c for c in texto if unicodedata.category(c) != 'Mn')

def ler_melhor_texto(imagem):
    orientacoes = (
        ('original', imagem),
        ('90_direita', cv2.rotate(imagem, cv2.ROTATE_90_CLOCKWISE)),
        ('90_esquerda', cv2.rotate(imagem, cv2.ROTATE_90_COUNTERCLOCKWISE)),
        ('180', cv2.rotate(imagem, cv2.ROTATE_180)),
    )
    melhor = ('', 'original', -1, -1.0)
    for nome, imagem_rotacionada in orientacoes:
        cinza = cv2.cvtColor(imagem_rotacionada, cv2.COLOR_BGR2GRAY)
        dados = pytesseract.image_to_data(
            cinza, lang='por', config='--psm 6', output_type=Output.DICT
        )
        palavras = []
        confiancas = []
        for palavra, conf in zip(dados['text'], dados['conf']):
            palavra = str(palavra).strip()
            if palavra:
                palavras.append(palavra)
                try:
                    conf = float(conf)
                    if conf >= 0:
                        confiancas.append(conf)
                except (ValueError, TypeError):
                    pass
        texto = ' '.join(palavras)
        texto_normalizado = normalizar_texto(texto)
        pontuacao = sum(termo in texto_normalizado for termo in PALAVRAS_CHAVE)
        confianca = sum(confiancas) / len(confiancas) if confiancas else 0.0
        if (pontuacao, confianca) > (melhor[2], melhor[3]):
            melhor = (texto, nome, pontuacao, confianca)
        if pontuacao >= 3:
            break
    return melhor

detector_rosto = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)
if detector_rosto.empty():
    raise RuntimeError('O detector de rosto do OpenCV não foi carregado.')
detector_qr = cv2.QRCodeDetector()

def analisar_imagem(caminho, classe):
    imagem = cv2.imread(str(caminho))
    if imagem is None:
        return None
    altura, largura = imagem.shape[:2]
    texto, orientacao, pontuacao, confianca = ler_melhor_texto(imagem)
    texto_normalizado = normalizar_texto(texto)
    cinza = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)
    rostos = detector_rosto.detectMultiScale(
        cinza, scaleFactor=1.1, minNeighbors=5, minSize=(25, 25)
    )
    try:
        tem_qr = int(detector_qr.detect(imagem)[0])
    except cv2.error:
        tem_qr = 0
    tem_filiacao = int('FILIACAO' in texto_normalizado)
    tem_identidade = int(any(x in texto_normalizado for x in (
        'CARTEIRA DE IDENTIDADE', 'IDENTIDADE NACIONAL', 'IDENTIFICACAO'
    )))
    return {
        'arquivo': caminho.name,
        'classe': classe,
        'largura_px': largura,
        'altura_px': altura,
        'proporcao_largura_altura': round(largura / altura, 3) if altura else 0,
        'orientacao_ocr_escolhida': orientacao,
        'pontuacao_palavras_chave': pontuacao,
        'confianca_media_ocr': round(confianca, 2),
        'quantidade_caracteres': len(texto_normalizado),
        'quantidade_palavras': len(texto.split()),
        'texto_extraido': texto.strip() if SALVAR_TEXTO_COMPLETO else '',
        'tem_texto_rg': int('REGISTRO GERAL' in texto_normalizado or re.search(r'\bRG\b', texto_normalizado) is not None),
        'tem_cpf': int('CPF' in texto_normalizado),
        'tem_republica': int('REPUBLICA' in texto_normalizado),
        'tem_brasil': int('BRASIL' in texto_normalizado),
        'tem_carteira_de_identidade': tem_identidade,
        'tem_data_nascimento': int('NASCIMENTO' in texto_normalizado or 'DATA DE NASC' in texto_normalizado),
        'tem_filiacao': tem_filiacao,
        'tem_filiacao_bilingue': int(tem_filiacao and ('PARENTAGE' in texto_normalizado or 'FILIATION' in texto_normalizado)),
        'possivel_cnh': int('CARTEIRA NACIONAL DE HABILITACAO' in texto_normalizado or re.search(r'\bCNH\b', texto_normalizado) is not None),
        'tem_qr_code': tem_qr,
        'tem_rosto_detectado': int(len(rostos) > 0),
        'nitidez_imagem': round(float(cv2.Laplacian(cinza, cv2.CV_64F).var()), 2),
    }

def executar_analise():
    linguas = pytesseract.get_languages(config='')
    if 'por' not in linguas:
        raise RuntimeError('Idioma português (por) não instalado no Tesseract OCR.')
    pasta_base = localizar_pasta_imagens()
    inicio = time.time()
    registros = []
    erros = 0
    for classe, pasta in NOMES_PASTAS.items():
        imagens = sorted(
            x for x in (pasta_base / pasta).iterdir()
            if x.is_file() and x.suffix.lower() in EXTENSOES_IMAGENS
        )
        print(f'Analisando classe {classe}: {len(imagens)} imagens')
        for caminho in imagens:
            try:
                registro = analisar_imagem(caminho, classe)
                if registro is None:
                    erros += 1
                else:
                    registros.append(registro)
            except (OSError, ValueError, RuntimeError, cv2.error) as erro:
                erros += 1
                print(f'Falha na análise de um arquivo da classe {classe}: {type(erro).__name__}')
    if not registros:
        raise RuntimeError('Nenhuma imagem pôde ser analisada.')
    destino = Path('/content/analise_documentos.csv') if Path('/content').is_dir() else Path('analise_documentos.csv')
    df = pd.DataFrame(registros)
    df.to_csv(destino, index=False, encoding='utf-8-sig')
    print(f'CSV privado criado em: {destino}')
    print(f'Imagens analisadas: {len(df)} | erros: {erros} | colunas: {len(df.columns)}')
    print('Distribuição por classe:')
    print(df['classe'].value_counts().to_string())
    print(f'Tempo aproximado: {(time.time()-inicio)/60:.1f} minutos')
    return df


In [ ]:
resultado_analise = executar_analise()
